# 🎓 Student Performance: Clustering + Linear Regression
**Pipeline:** K-Means Clustering → Assign Cluster Labels → Linear Regression per Cluster

---
## STEP 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

print("✅ All libraries imported successfully!")

---
## STEP 2: Load Dataset

In [ ]:
# Upload your CSV manually in Colab first, then load it
df = pd.read_csv("bd_class5_students_dataset.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

---
## STEP 3: Select Features for Clustering

In [ ]:
features = df[[
    "Bangla",
    "English",
    "Math",
    "Science",
    "Religion",
    "Social_Science"
]]

print("✅ Features selected:")
print(features.describe())

---
## STEP 4: Normalize Data

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

print("✅ Data normalized (StandardScaler applied)")
print("Scaled shape:", scaled_features.shape)

---
## STEP 5: Find Best K using Elbow Method

In [ ]:
inertia = []
K_range = range(1, 10)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(scaled_features)
    inertia.append(model.inertia_)

# Plot Elbow Graph
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker='o', color='steelblue', linewidth=2)
plt.title("Elbow Method to Find Optimal K", fontsize=14)
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(K_range)
plt.grid(True)
plt.tight_layout()
plt.show()

---
## STEP 6: Apply K-Means Clustering (K = 3)

In [ ]:
# Choose K = 3 based on elbow method (adjust if needed)
K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(scaled_features)

print("✅ K-Means applied with K =", K)
print("\nCluster distribution:")
print(df["Cluster"].value_counts().sort_index())
print("\nSample Name → Cluster mapping:")
print(df[["Name", "Cluster"]].head(10))

---
## STEP 7: Visualize Clusters using PCA

In [ ]:
pca = PCA(n_components=2)
pca_features = pca.fit_transform(scaled_features)
df["PCA1"] = pca_features[:, 0]
df["PCA2"] = pca_features[:, 1]

colors = ["#e74c3c", "#2ecc71", "#3498db"]
plt.figure(figsize=(9, 6))
for cluster_id in sorted(df["Cluster"].unique()):
    subset = df[df["Cluster"] == cluster_id]
    plt.scatter(
        subset["PCA1"], subset["PCA2"],
        label=f"Cluster {cluster_id}",
        color=colors[cluster_id],
        alpha=0.7, s=80, edgecolors='k', linewidths=0.4
    )

plt.title("Student Clusters (PCA Projection)", fontsize=14)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title="Cluster")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## STEP 8: Analyze Clusters (Average Marks per Cluster)

In [ ]:
subject_cols = ["Bangla", "English", "Math", "Science", "Religion", "Social_Science"]

cluster_analysis = df.groupby("Cluster")[subject_cols].mean().round(2)
print("\n📊 Cluster-Wise Average Marks:\n")
print(cluster_analysis)

# Heatmap visualization
plt.figure(figsize=(9, 4))
sns.heatmap(
    cluster_analysis,
    annot=True, fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.5
)
plt.title("Average Marks per Cluster", fontsize=14)
plt.tight_layout()
plt.show()

---
## STEP 9: Add Total Marks Column (Regression Target)

In [ ]:
# Create a Total Marks column as the regression target
df["Total"] = df[subject_cols].sum(axis=1)

print("✅ Total Marks column created")
print(df[["Name", "Cluster", "Total"]].head(10))

---
## STEP 10: Linear Regression on Each Cluster
We train a **separate Linear Regression model** for each cluster.  
**Target:** `Total` marks | **Features:** individual subject scores

In [ ]:
regression_results = {}

for cluster_id in sorted(df["Cluster"].unique()):
    cluster_df = df[df["Cluster"] == cluster_id].copy()

    X = cluster_df[subject_cols]
    y = cluster_df["Total"]

    # Train-Test Split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Fit Linear Regression
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    regression_results[cluster_id] = {
        "model": lr,
        "MSE": round(mse, 4),
        "R2":  round(r2,  4),
        "coefficients": dict(zip(subject_cols, lr.coef_.round(4))),
        "intercept": round(lr.intercept_, 4)
    }

    print(f"\n{'='*45}")
    print(f"  Cluster {cluster_id}  (n = {len(cluster_df)} students)")
    print(f"{'='*45}")
    print(f"  R² Score : {r2:.4f}")
    print(f"  MSE      : {mse:.4f}")
    print(f"  Intercept: {lr.intercept_:.4f}")
    print("  Coefficients:")
    for subj, coef in zip(subject_cols, lr.coef_):
        print(f"    {subj:<15}: {coef:.4f}")

---
## STEP 11: Visualize Actual vs Predicted (per Cluster)

In [ ]:
fig, axes = plt.subplots(1, K, figsize=(6 * K, 5))

for cluster_id in sorted(df["Cluster"].unique()):
    cluster_df = df[df["Cluster"] == cluster_id].copy()
    X = cluster_df[subject_cols]
    y = cluster_df["Total"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    lr = regression_results[cluster_id]["model"]
    y_pred = lr.predict(X_test)

    ax = axes[cluster_id]
    ax.scatter(y_test, y_pred, alpha=0.7, color=colors[cluster_id],
               edgecolors='k', linewidths=0.4, s=70)

    # Perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect Fit')

    r2  = regression_results[cluster_id]["R2"]
    mse = regression_results[cluster_id]["MSE"]
    ax.set_title(f"Cluster {cluster_id} — Actual vs Predicted\nR²={r2}  MSE={mse}", fontsize=12)
    ax.set_xlabel("Actual Total")
    ax.set_ylabel("Predicted Total")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle("Linear Regression: Actual vs Predicted per Cluster", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## STEP 12: Regression Performance Summary

In [ ]:
summary = pd.DataFrame({
    "Cluster": list(regression_results.keys()),
    "R2_Score": [v["R2"]  for v in regression_results.values()],
    "MSE":      [v["MSE"] for v in regression_results.values()],
    "Students": [len(df[df["Cluster"] == c]) for c in regression_results.keys()]
})

print("\n📋 Regression Performance Summary:\n")
print(summary.to_string(index=False))

# Bar chart of R²
plt.figure(figsize=(6, 4))
plt.bar(
    [f"Cluster {c}" for c in summary["Cluster"]],
    summary["R2_Score"],
    color=colors[:K], edgecolor='k'
)
plt.title("R² Score per Cluster", fontsize=13)
plt.ylabel("R² Score")
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()